%%bash
# 1. Generate the Private Key (.p8 file)
openssl genrsa 2048 | openssl pkcs8 -topk8 -inform PEM -out /home/jovyan/work/rsa_key.p8 -nocrypt

# 2. Generate the Public Key
openssl rsa -in /home/jovyan/work/rsa_key.p8 -pubout -out /home/jovyan/work/rsa_key.pub

# 3. Print the Public Key (you will need to copy this!)
echo -e "\n=== COPY THE TEXT BELOW THIS LINE ==="
grep -v "BEGIN PUBLIC" /home/jovyan/work/rsa_key.pub | grep -v "END PUBLIC" | tr -d '\r\n'
echo -e "\n=== COPY THE TEXT ABOVE THIS LINE ==="

In [11]:
%%writefile /home/jovyan/work/.env
SNOWFLAKE_ACCOUNT=MPLXIUW-YA71548
SNOWFLAKE_USER=MANS3
SNOWFLAKE_PRIVATE_KEY_PATH=/home/jovyan/work/rsa_key.p8
SNOWFLAKE_DATABASE=IPL_DB
SNOWFLAKE_SCHEMA=RAW
SNOWFLAKE_WAREHOUSE=IPL_WH
SNOWFLAKE_ROLE=ACCOUNTADMIN

Overwriting /home/jovyan/work/.env


In [12]:
# Cell — Install
import subprocess
subprocess.run(["pip", "install", "snowflake-connector-python", 
                "pandas", "-q"])
print("✅ Installed")

✅ Installed


In [14]:
# Cell — Load IPL Parquet → Snowflake (Using .p8 Token)
import snowflake.connector
import pandas as pd
from dotenv import load_dotenv
import os

# These are required to read the .p8 token
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, dsa
from cryptography.hazmat.primitives import serialization

# 1. Load the variables from the .env file we just created
load_dotenv("/home/jovyan/work/.env", override=True)

# 2. Securely read the .p8 Private Key file
private_key_file = os.getenv("SNOWFLAKE_PRIVATE_KEY_PATH")

with open(private_key_file, "rb") as key:
    p_key = serialization.load_pem_private_key(
        key.read(),
        password=None, # We generated it without a password earlier
        backend=default_backend()
    )

pkb = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# 3. Connect to Snowflake using the token (No Duo push needed!)
conn = snowflake.connector.connect(
    account     = os.getenv("SNOWFLAKE_ACCOUNT"),
    user        = os.getenv("SNOWFLAKE_USER"),
    private_key = pkb, # <-- Passing the token here
    role        = os.getenv("SNOWFLAKE_ROLE"),
    database    = os.getenv("SNOWFLAKE_DATABASE"),
    schema      = os.getenv("SNOWFLAKE_SCHEMA"),
    warehouse   = os.getenv("SNOWFLAKE_WAREHOUSE")
)
cur = conn.cursor()
print("✅ Snowflake successfully connected via .p8 Token!")

# ---------------------------------------------------------
# 4. Load your Parquet Data
# ---------------------------------------------------------
# Read parquet outputs from Day 2
df_wins   = pd.read_parquet("output/team_wins")
df_season = pd.read_parquet("output/season_summary")
df_toss   = pd.read_parquet("output/toss_analysis")

# 🔥 FIX: Clean the 'season' column to be standard integers
# This takes '2007/08', splits it at the '/', takes '2007', and turns it into a real integer
df_season['season'] = df_season['season'].astype(str).str.split('/').str[0].astype(int)

print(f"  team_wins    : {len(df_wins)} rows")
print(f"  season_summary : {len(df_season)} rows")
print(f"  toss_analysis  : {len(df_toss)} rows")
# ... (rest of your script remains exactly the same)

def load_df(df, table_ddl, table_name, cur):
    cur.execute(table_ddl)
    cur.execute(f"TRUNCATE TABLE {table_name}")
    cols = ", ".join(df.columns)
    ph   = ", ".join(["%s"] * len(df.columns))
    rows = [tuple(r) for r in df.itertuples(index=False)]
    cur.executemany(f"INSERT INTO {table_name} ({cols}) VALUES ({ph})", rows)
    print(f"  ✅ {table_name}: {len(rows)} rows")

# Load all 3 tables
load_df(df_wins,
    """CREATE OR REPLACE TABLE IPL_DB.RAW.TEAM_WINS (
        winner VARCHAR, total_wins INTEGER)""",
    "IPL_DB.RAW.TEAM_WINS", cur)

load_df(df_season,
    """CREATE OR REPLACE TABLE IPL_DB.RAW.SEASON_SUMMARY (
        season INTEGER, matches_played INTEGER)""",
    "IPL_DB.RAW.SEASON_SUMMARY", cur)

load_df(df_toss,
    """CREATE OR REPLACE TABLE IPL_DB.RAW.TOSS_ANALYSIS (
        toss_decision VARCHAR, total_matches INTEGER)""",
    "IPL_DB.RAW.TOSS_ANALYSIS", cur)

# Verify
print("\n[VERIFY] Row counts:")
for t in ["TEAM_WINS", "SEASON_SUMMARY", "TOSS_ANALYSIS"]:
    cur.execute(f"SELECT COUNT(*) FROM IPL_DB.RAW.{t}")
    print(f"  IPL_DB.RAW.{t}: {cur.fetchone()[0]} rows ✅")

cur.close()
conn.close()
print("\n🎉 Pipeline complete!")

✅ Snowflake successfully connected via .p8 Token!
  team_wins    : 20 rows
  season_summary : 17 rows
  toss_analysis  : 2 rows
  ✅ IPL_DB.RAW.TEAM_WINS: 20 rows
  ✅ IPL_DB.RAW.SEASON_SUMMARY: 17 rows
  ✅ IPL_DB.RAW.TOSS_ANALYSIS: 2 rows

[VERIFY] Row counts:
  IPL_DB.RAW.TEAM_WINS: 20 rows ✅
  IPL_DB.RAW.SEASON_SUMMARY: 17 rows ✅
  IPL_DB.RAW.TOSS_ANALYSIS: 2 rows ✅

🎉 Pipeline complete!
